# Week 6 — CampusIQ Final Integrated Pipeline
## Academic Policy Retrieval-Augmented Generation (RAG)

**Purpose:** Demonstrate the end-to-end policy retrieval workflow using the JNTUH R22 B.Tech Academic Regulations PDF.

### Pipeline
1. Read the policy PDF and extract text.
2. Clean text and split it into overlapping chunks.
3. Build semantic embeddings.
4. Clean and analyze a student question.
5. Retrieve relevant policy chunks.
6. Assemble a grounded prompt with source context.
7. Run simulated integration checks and summarize their results.

**Scope note:** The current project builds a prompt; it does not call an answer-generating language model. Therefore, this notebook evaluates retrieval and prompt construction, not the factual accuracy of generated answers. Run all cells from top to bottom in the repository root.


In [1]:
from pathlib import Path
import sys
import json
import time
import pandas as pd

ROOT = Path.cwd()
if not (ROOT / "src").exists():
    # Allows running from the notebooks/ directory as well.
    ROOT = ROOT.parent

SRC = ROOT / "src"
DATA_RAW = ROOT / "data" / "raw"
DATA_PROCESSED = ROOT / "data" / "processed"
REPORTS = ROOT / "reports" / "week6"
REPORTS.mkdir(parents=True, exist_ok=True)

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

print("Project root:", ROOT)
print("Raw data directory exists:", DATA_RAW.exists())
print("Source directory exists:", SRC.exists())


Project root: /workspaces/campusiq-academic-policy-rag
Raw data directory exists: True
Source directory exists: True


In [2]:
from data_processing import process_all_pdfs
from query_understanding import analyze_query
from vector_store import PolicyVectorStore
from rag_pipeline import CampusIQRAG
from evaluation import calculate_retrieval_accuracy, evaluate_queries

print("Project modules imported successfully.")


/home/codespace/.local/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Project modules imported successfully.


## 1. Load and preprocess the source PDF

The existing project processor reads PDFs from `data/raw/`, cleans whitespace, and creates chunks of up to 1,000 characters with 150-character overlap. The code below also writes a fresh CSV so the notebook uses the current PDF rather than a stale demonstration file.


In [3]:
documents = process_all_pdfs()

if not documents:
    raise FileNotFoundError(
        "No PDF chunks were created. Confirm that the academic regulations PDF is in data/raw/."
    )

required_fields = {"source", "chunk_id", "text"}
missing_fields = required_fields - set(documents[0].keys())
if missing_fields:
    raise ValueError(f"Processed documents are missing fields: {missing_fields}")

chunks_df = pd.DataFrame(documents)
chunks_df = chunks_df.dropna(subset=["text"])
chunks_df["text"] = chunks_df["text"].astype(str)
chunks_df = chunks_df[chunks_df["text"].str.strip().ne("")]
chunks_df = chunks_df.drop_duplicates(subset=["source", "chunk_id"]).reset_index(drop=True)

DATA_PROCESSED.mkdir(parents=True, exist_ok=True)
chunks_csv = DATA_PROCESSED / "jntuh_policy_chunks.csv"
chunks_df.to_csv(chunks_csv, index=False)

print("Chunks:", len(chunks_df))
print("Unique sources:", chunks_df["source"].nunique())
print("CSV saved to:", chunks_csv)
display(chunks_df[["source", "chunk_id"]].head())


Processing: R22B.Tech.AcademicRegulations.pdf
Chunks: 62
Unique sources: 1
CSV saved to: /workspaces/campusiq-academic-policy-rag/data/processed/jntuh_policy_chunks.csv


,source,chunk_id
0,R22B.Tech.AcademicRegulations.pdf,0
1,R22B.Tech.AcademicRegulations.pdf,1
2,R22B.Tech.AcademicRegulations.pdf,2
3,R22B.Tech.AcademicRegulations.pdf,3
4,R22B.Tech.AcademicRegulations.pdf,4


## 2. Inspect dataset quality

These checks describe the processed chunks; they do not verify the legal or academic correctness of every extracted statement.


In [4]:
quality_summary = {
    "row_count": int(len(chunks_df)),
    "missing_text": int(chunks_df["text"].isna().sum()),
    "empty_text": int(chunks_df["text"].str.strip().eq("").sum()),
    "duplicate_source_chunk_pairs": int(chunks_df.duplicated(["source", "chunk_id"]).sum()),
    "mean_characters": float(chunks_df["text"].str.len().mean()),
    "min_characters": int(chunks_df["text"].str.len().min()),
    "max_characters": int(chunks_df["text"].str.len().max()),
}
quality_summary


{'row_count': 62,
 'missing_text': 0,
 'empty_text': 0,
 'duplicate_source_chunk_pairs': 0,
 'mean_characters': 998.1290322580645,
 'min_characters': 903,
 'max_characters': 1000}

## 3. Build the semantic retrieval index

The embedding model may download model files the first time it runs, so an internet connection may be required. If model loading fails, record the error rather than claiming that retrieval tests passed.


In [5]:
rag = CampusIQRAG()
rag.load_documents(chunks_df.to_dict(orient="records"))
print("Indexed documents:", len(rag.vector_store.documents))
print("Embedding matrix shape:", getattr(rag.vector_store.embeddings, "shape", None))


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 15224.60it/s]


Indexed documents: 62
Embedding matrix shape: (62, 384)


## 4. Try a student question

Inspect the retrieved chunks and their source names. Similarity scores are ranking signals, not probabilities that a result is correct.


In [6]:
sample_query = "How many credits are required to complete the B.Tech programme?"
result = rag.run(sample_query, top_k=5)

print("Question:", result["query"])
print("Query analysis:", result["query_analysis"])
print("\nRetrieved evidence:")
for rank, item in enumerate(result["retrieved_documents"], start=1):
    print(f"\n[{rank}] source={item.get('source')} chunk_id={item.get('chunk_id')} score={item.get('score', float('nan')):.4f}")
    print(item.get("text", "")[:700])

print("\nGrounded prompt preview:")
print(result["prompt"][:2500])


Question: How many credits are required to complete the B.Tech programme?
Query analysis: {'original_query': 'How many credits are required to complete the B.Tech programme?', 'cleaned_query': 'how many credits are required to complete the b tech programme?', 'categories': ['grading'], 'has_category': True}

Retrieved evidence:

[1] source=R22B.Tech.AcademicRegulations.pdf chunk_id=1 score=0.7405
r on the basis of any other
order of merit approved by the University, subject to reservations as prescribed by the
government from time to time.
2.2 The medium of instructions for the entire undergraduate programme in Engineering &
Technology will be English only.
3.0 B.Tech. Programme Structure
3.1 A student after securing admission shall complete the B.Tech. programme in a minimum
period of four academic years (8 semesters), and a maximum period of eight academic
years (16 semesters) starting from the date of commencement of first year first semester,
failing which student shall for feit se

## 5. Simulated integration tests

These tests check software behavior and retrieval plumbing. They do **not** establish that every policy answer is correct. The expected-source checks below use the source PDF name, which is suitable for an integration smoke test but is not a fine-grained gold-standard relevance label.


In [7]:
test_rows = []

def record_test(name, passed, details=""):
    test_rows.append({
        "test": name,
        "status": "PASS" if bool(passed) else "FAIL",
        "details": str(details)
    })

# Test A: processing produced non-empty chunks.
record_test("PDF preprocessing produces chunks", len(chunks_df) > 0, f"{len(chunks_df)} chunks")

# Test B: chunk schema is available.
record_test("Required chunk fields exist",
            required_fields.issubset(set(chunks_df.columns)),
            f"columns={chunks_df.columns.tolist()}")

# Test C: query cleaning and category detection run.
analysis = analyze_query("  What attendance is required for exams?  ")
record_test("Query analysis returns cleaned query",
            bool(analysis.get("cleaned_query")) and analysis.get("cleaned_query") == analysis.get("cleaned_query", "").strip(),
            analysis)

# Test D: retrieval returns at least one chunk for a non-empty query.
retrieved = rag.retrieve(sample_query, top_k=5)
record_test("Semantic retrieval returns results", len(retrieved) > 0, f"{len(retrieved)} results")

# Test E: retrieved chunks contain source and text fields.
schema_ok = all("source" in d and "text" in d for d in retrieved)
record_test("Retrieved results preserve source and text", schema_ok, f"{len(retrieved)} results checked")

# Test F: context and prompt are constructed.
record_test("Context and grounded prompt are constructed",
            bool(result.get("context")) and "Academic policy context:" in result.get("prompt", ""),
            f"context_chars={len(result.get('context', ''))}")

# Test G: empty result behavior returns a safe message.
empty_context = rag.build_context([])
record_test("Empty retrieval has safe fallback",
            "No relevant policy information" in empty_context, empty_context)

# Test H: evaluation helper handles a matching source.
toy_results = [{"source": "policy.pdf", "text": "example"}]
record_test("Evaluation helper detects expected source",
            calculate_retrieval_accuracy(toy_results, "policy.pdf") == 1.0,
            "matching source returned 1.0")

test_df = pd.DataFrame(test_rows)
display(test_df)


,test,status,details
0,PDF preprocessing produces chunks,PASS,62 chunks
1,Required chunk fields exist,PASS,"columns=['source', 'chunk_id', 'text']"
2,Query analysis returns cleaned query,PASS,{'original_query': ' What attendance is requi...
3,Semantic retrieval returns results,PASS,5 results
4,Retrieved results preserve source and text,PASS,5 results checked
5,Context and grounded prompt are constructed,PASS,context_chars=5237
6,Empty retrieval has safe fallback,PASS,No relevant policy information was found.
7,Evaluation helper detects expected source,PASS,matching source returned 1.0


In [8]:
test_csv = REPORTS / "integration_test_results.csv"
test_df.to_csv(test_csv, index=False)

passed = int((test_df["status"] == "PASS").sum())
failed = int((test_df["status"] == "FAIL").sum())
summary = {
    "test_count": int(len(test_df)),
    "passed": passed,
    "failed": failed,
    "pass_rate": float(passed / len(test_df)) if len(test_df) else 0.0,
    "dataset_chunks": int(len(chunks_df)),
    "source_pdf_count": int(chunks_df["source"].nunique()),
    "answer_generation_model_used": False,
    "note": "Integration smoke tests only; they do not validate generated answers or independent policy labels."
}
summary_path = REPORTS / "integration_test_summary.json"
summary_path.write_text(json.dumps(summary, indent=2), encoding="utf-8")

print(json.dumps(summary, indent=2))
print("Saved:", test_csv)
print("Saved:", summary_path)

if failed:
    print("WARNING: One or more tests failed. Inspect the test table before submission.")


{
  "test_count": 8,
  "passed": 8,
  "failed": 0,
  "pass_rate": 1.0,
  "dataset_chunks": 62,
  "source_pdf_count": 1,
  "answer_generation_model_used": false,
  "note": "Integration smoke tests only; they do not validate generated answers or independent policy labels."
}
Saved: /workspaces/campusiq-academic-policy-rag/reports/week6/integration_test_results.csv
Saved: /workspaces/campusiq-academic-policy-rag/reports/week6/integration_test_summary.json


## 6. Mock deployment strategy

### Runtime environment
- Python environment with packages from `requirements.txt`.
- Store source policy PDFs in `data/raw/`; regenerate chunks when policy documents change.
- Run the notebook or expose the same retrieval flow through a Streamlit interface.
- Keep model downloads and caches in the deployment environment; avoid downloading models on every user request.

### Request flow
1. Validate that the student question is not empty.
2. Normalize and classify the question.
3. Embed the question and rank policy chunks.
4. Return the relevant source/chunk references and a grounded prompt.
5. In a future extension, pass the prompt to an approved language model, then validate and display citations. This final answer-generation step is not implemented in the current pipeline.

### Scalability
- Cache the embedding model and index between requests.
- Rebuild embeddings only when policy content changes.
- For larger collections, use a persistent vector database and monitor retrieval latency.
- Keep document versions and ingestion timestamps for traceability.

### Monitoring and safety
- Log latency, empty retrievals, top similarity scores, source IDs, and application errors without storing unnecessary personal data.
- Review retrieval failures and update test queries.
- Clearly show source documents and tell students to confirm consequential decisions with the university.
- Do not invent attendance thresholds, fees, dates, or eligibility rules when the source does not support them.


## 7. User guide and troubleshooting

1. Put the official policy PDF in `data/raw/`.
2. Open this notebook from the repository and run all cells in order.
3. Review the dataset-quality summary and retrieved chunks.
4. Inspect `reports/week6/integration_test_results.csv` and `integration_test_summary.json`.
5. Commit the notebook and generated reports only after reviewing them.

**Common problems**
- `ModuleNotFoundError`: install project dependencies from `requirements.txt` in the active environment.
- Embedding model download error: check network access and retry model loading.
- Zero chunks: check the PDF path and whether text extraction is supported by the PDF.
- Irrelevant retrieval: try different test questions, inspect source text, and evaluate against manually reviewed expected chunks.
- Missing final natural-language answer: expected in this version; the current pipeline builds a prompt but does not call an answer-generation model.


## 8. Results and limitations

The notebook creates measured results when its cells are run. Do not report pass counts, latency, or retrieval accuracy before executing it in the target environment. The existing classifier metrics from earlier weeks are a separate experiment and should not be presented as RAG retrieval metrics.

Limitations:
- A single PDF and a small number of integration smoke tests do not prove production readiness.
- Source-level checks are weaker than human-reviewed query-to-chunk relevance labels.
- The current code does not generate an answer using a language model.
- OCR-heavy or scanned policy PDFs may require additional preprocessing.
- Integration results should be rerun after changes to documents, dependencies, or retrieval logic.
